In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Code Evaluation for Circuit Analysis

This notebook evaluates the code implementation in `/net/scratch2/smallyan/filter_eval` for circuit analysis.

## 1. Setup and Initial Exploration

In [2]:
# First, let's explore the repository structure
import os
import subprocess

repo_path = "/net/scratch2/smallyan/filter_eval"
print("Repository structure:")
for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories and common non-essential dirs
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__', 'node_modules', '.git']]
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

Repository structure:
filter_eval/
  plan.md
  documentation.pdf
  env.yml
  demo.ipynb
  env_demo.yml
  globals.yml
  run_finetuning.py
  CodeWalkthrough.md
  conda_env.yml
  run_monitor.py
  .gitignore
  data/
    LLM_CACHE/
      claude_cache/
      gpt4o_cache/
  test_suite/
    test_05_rome_all.py
    test_04_rome_mixed.py
    test_03_synth_entities.py
    test_02_synth_real.py
    test_01_real_entities.py
  evaluation/
    self_matching.ipynb
    generalization_eval.ipynb
    consistency_evaluation.json
    generalization_eval_summary.json
    replications/
      documentation_replication.md
      evaluation_replication.md
      replication.ipynb
      self_replication_evaluation.json
    replication_eval/
      documentation_eval_summary.json
      documentation_evaluation_summary.md
  data_save/
    deduction/
      logic_templates.json
      topics.json
    selection/
      landmarks.json
      nationality.json
      rhymes.json
      profession.json
      objects.json
    pro

      atheletes/
        tennis.json
        basketball.json
        golf.json
        soccer.json
        cricket.json
        baseball.json
      profession/
        musicians.json
        politicians.json
        chefs.json
        actors.json
        scientists.json
        writers.json
    counting/
      fruits.json
  src/
    trace.py
    dataset.py
    tokens.py
    functional.py
    plotting.py
    evaluation.py
    models.py
    ablation.py
    data.py
    attention.py
    __init__.py
    globals.py
    hooking/
      llama_attention.py
      __init__.py
    operators/
      utils.py
      operators.py
      estimators.py
      baselines.py
      editor.py
    utils/
      typing.py
      env_utils.py
      metrics.py
      tokenization_utils.py
      training_utils.py
      oracle_llms.py
      __init__.py
      experiment_utils.py
      logging_utils.py
    rome/
      tok_dataset.py
      rome_hparams.py
      repr_tools.py
      compute_v.py
      compute_u.py
      READM

      Llama-3.3-70B-Instruct/
        filter_vs_function.pdf
        filter_vs_concept.pdf
        svd_qk.pdf
        ov_patching_filter_heads.pdf
        aie/
          objects_aie.pdf
          objects-sliced.pdf
          objects.pdf
          svd_qk.pdf
          objects_aie_wo_sparsity.pdf
          objects_attn.pdf
          objects-slided.pdf
        raw/
          probe_performance.json
          attention_pattern.json
          aie_per_head.json
        logit_lens_contribution/
          template_3_mcq.pdf
          template_3.pdf
        residual/
          template_2.pdf
          template_3.pdf
          residual_2_flag_ablate.pdf
          residual_3.pdf
          residual_3_flag_ablate.pdf
          residual_2.pdf
    checking_predicate_generalization/
      pred_gen.ipynb
      select_one.ipynb
      select_last.ipynb
      select_first.ipynb
  results/
  scripts/
    locate_via_attention_behavior.py
    __init__.py
    cache_thinking_process.py
    dcm_on_svd_q_proj.py


In [3]:
# Read the CodeWalkthrough.md to understand the project structure and main analysis
with open(os.path.join(repo_path, "CodeWalkthrough.md"), 'r') as f:
    codewalk_content = f.read()
print(codewalk_content)

# LLMs Process Lists With General Filter Heads
### [Project Website](https://filter.baulab.info/) | [Arxiv Preprint](https://arxiv.org/pdf/2510.26784)

This repository contains code, data, and visualizations for the paper "LLMs Process Lists With General Filter Heads".

How does an LLM perform *filtering* operation over a list of items? We find that a small set of specialized attention heads, which we call *filter heads*, are responsible for this across a range of different situations. The query states of these heads encode a compact representation of the filtering criterion (the *predicate*), which can be transported to a different context to trigger the execution of the same filtering operation on a different list of items, presented in a different format, language, even different tasks.

<p align="center">
<img src="notebooks/figures/fig_1_sliced-crop-1.png" style="width:100%;"/>
</p> 

Checkout [filter.baulab.info](http://filter.baulab.info) for more details.


## Setup

All code i

In [4]:
# Read the plan.md to understand the project goals
with open(os.path.join(repo_path, "plan.md"), 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Investigate the mechanisms underlying list-processing tasks in LLMs to understand how they encode and execute filtering operations that mirror the generic 'filter' function of functional programming.

## Hypothesis
1. A small number of attention heads (filter heads) encode a compact representation of the filtering predicate in their query states at certain tokens.
2. The predicate representation in filter heads is general and portable: it can be extracted and reapplied to execute the same filtering operation on different collections, formats, languages, or tasks.
3. Transformer LMs can exploit a different strategy for filtering: eagerly evaluating if an item satisfies the predicate and storing this intermediate result as a flag directly in the item representations.

## Methodology
1. Apply causal mediation analysis using activation patching on diverse list-processing tasks. Identify filter heads by patching query states (qsrc) from a source prompt to a destination p

## Understanding the Project

Based on the CodeWalkthrough.md and plan.md, the project investigates "filter heads" in LLMs - specialized attention heads responsible for list-processing/filtering operations.

**Key Components to Evaluate:**
1. **demo.ipynb** - Main demonstration notebook mentioned in CodeWalkthrough
2. **scripts/locate_selection_heads.py** - Script for locating filter heads
3. **src/** - Core source modules (dataset, attention, ablation, etc.)
4. **notebooks/** - Analysis notebooks

Let me first evaluate the demo.ipynb as it's the primary entry point mentioned in the codewalk.

In [5]:
# First, let's check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU count: 1
Current device: 0
Device name: NVIDIA A100 80GB PCIe


In [6]:
# Read the demo.ipynb to understand its structure
import json
demo_path = os.path.join(repo_path, "demo.ipynb")
with open(demo_path, 'r') as f:
    demo_nb = json.load(f)

print(f"Number of cells: {len(demo_nb['cells'])}")
print("\nCell types and first lines:")
for i, cell in enumerate(demo_nb['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])[:100].replace('\n', ' ')
    print(f"Cell {i}: [{cell_type}] {source}...")

Number of cells: 19

Cell types and first lines:
Cell 0: [code] %load_ext autoreload %autoreload 2...
Cell 1: [code] import torch import transformers from src.models import ModelandTokenizer  print(f"{torch.__version_...
Cell 2: [code] # select one of the filter heads if model_key == "meta-llama/Llama-3.3-70B-Instruct":     layer_idx,...
Cell 3: [markdown] ## Checking the behavior of a filter head on one example ...
Cell 4: [code] from src.selection.data import SelectOneTask from typing import Literal import os # from src.utils i...
Cell 5: [code] sample = select_task.get_random_sample(     mt = mt,     option_style=option_style,     prompt_templ...
Cell 6: [code] from src.selection.functional import verify_head_patterns  #! the select head is good but not 100% p...
Cell 7: [markdown] ## Patching the query state to transfer the predicate  See Figure 1 in the paper.  <p align="center"...
Cell 8: [code] from src.selection.data import get_counterfactual_samples_within_task  source_sample,

In [7]:
# Let's read the full content of each code cell in the demo notebook
for i, cell in enumerate(demo_nb['cells']):
    if cell['cell_type'] == 'code':
        source = ''.join(cell['source'])
        print(f"\n{'='*60}")
        print(f"CELL {i} (code):")
        print(f"{'='*60}")
        print(source)


CELL 0 (code):
%load_ext autoreload
%autoreload 2

CELL 1 (code):
import torch
import transformers
from src.models import ModelandTokenizer

print(f"{torch.__version__=}, {torch.version.cuda=}")
print(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
print(f"{transformers.__version__=}")

model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "google/gemma-2-27b-it"

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

CELL 2 (code):
# select one of the filter heads
if model_key == "meta-llama/Llama-3.3-70B-Instruct":
    layer_idx, head_idx = 35, 19
elif model_key == "google/gemma-2-27b-it":
    layer_idx, head_idx = 29, 3
else:
    raise ValueError("For other models you need to localize the heads first. Check scripts/locate_selection_heads.py")

CELL 4 (code):
from src.selection.data import SelectOneTask
from typing import Literal
import o

## 2. Evaluating demo.ipynb

The demo notebook has 19 cells total (17 code cells, 2 markdown cells). Let me run each code cell and evaluate it.

In [8]:
# Change to the repo directory to run demo notebook cells
os.chdir(repo_path)
print(f"Changed to: {os.getcwd()}")

# Initialize evaluation tracking
evaluation_results = []

# Cell 0: autoreload setup
cell_0_result = {
    "cell_id": "demo.ipynb Cell 0",
    "description": "autoreload magic commands",
    "runnable": None,
    "correct_implementation": None,
    "redundant": None,
    "irrelevant": None,
    "notes": ""
}

# These are jupyter magic commands - we'll mark as runnable (they don't apply in this context)
cell_0_result["runnable"] = "Y"
cell_0_result["correct_implementation"] = "Y"
cell_0_result["redundant"] = "N"
cell_0_result["irrelevant"] = "N"  # autoreload is useful for development
cell_0_result["notes"] = "Jupyter magic commands for autoreload"
evaluation_results.append(cell_0_result)
print("Cell 0: PASS (autoreload magic commands)")

Changed to: /net/scratch2/smallyan/filter_eval
Cell 0: PASS (autoreload magic commands)


In [9]:
# Cell 1: Import and model loading
# This is the most critical cell - loading the model

cell_1_result = {
    "cell_id": "demo.ipynb Cell 1",
    "description": "Import libraries and load model",
    "runnable": None,
    "correct_implementation": None,
    "redundant": None,
    "irrelevant": None,
    "notes": ""
}

try:
    import torch
    import transformers
    from src.models import ModelandTokenizer

    print(f"{torch.__version__=}, {torch.version.cuda=}")
    print(
        f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
    )
    print(f"{transformers.__version__=}")

    model_key = "meta-llama/Llama-3.3-70B-Instruct"
    # model_key = "google/gemma-2-27b-it"

    mt = ModelandTokenizer(
        model_key=model_key,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    
    cell_1_result["runnable"] = "Y"
    cell_1_result["correct_implementation"] = "Y"
    cell_1_result["redundant"] = "N"
    cell_1_result["irrelevant"] = "N"
    cell_1_result["notes"] = "Model loaded successfully"
    print("Cell 1: PASS")
except Exception as e:
    cell_1_result["runnable"] = "N"
    cell_1_result["correct_implementation"] = "N"
    cell_1_result["redundant"] = "N"
    cell_1_result["irrelevant"] = "N"
    cell_1_result["notes"] = f"Error: {str(e)}"
    print(f"Cell 1: FAIL - {e}")

evaluation_results.append(cell_1_result)

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


meta-llama/Llama-3.3-70B-Instruct not found in /net/projects/chai-lab/shared_models
If not found in cache, model will be downloaded from HuggingFace to cache directory


torch.__version__='2.7.1+cu118', torch.version.cuda='11.8'
torch.cuda.is_available()=True, torch.cuda.device_count()=1, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
transformers.__version__='4.57.3'


`torch_dtype` is deprecated! Use `dtype` instead!


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [10]:
# Check if model loaded successfully
print(f"Model loaded: {mt.model_key}")
print(f"Model type: {type(mt.model)}")
print(f"Tokenizer type: {type(mt.tokenizer)}")

In [11]:
# Wait for model loading to complete and verify
import time
time.sleep(5)
print(f"Model loaded: {mt.model_key}")
print(f"Model type: {type(mt.model)}")
print(f"Tokenizer type: {type(mt.tokenizer)}")

In [12]:
# Let's check if mt is defined and has the expected attributes
try:
    print(f"mt exists: {mt is not None}")
    print(f"mt.model_key: {mt.model_key}")
except Exception as e:
    print(f"Error accessing mt: {e}")

In [13]:
# Check the status
import sys
print("Checking model status...")
print(f"mt defined: {'mt' in dir()}")
if 'mt' in dir():
    print(f"mt type: {type(mt)}")
    print(f"mt.model_key: {getattr(mt, 'model_key', 'not found')}")

In [14]:
print("Test output")

In [15]:
# Let me check if the kernel is responsive
import sys
print("Kernel is responsive")
print(f"Python version: {sys.version}")